# Phase 2S - Segmentation with real ground truth### Retrain the U-Net on MSD Task06_Lung, report Dice / IoU on held-out patients**Why this notebook exists.** IQ-OTH/NCCD ships class labels only. It has noper-pixel annotations, so Dice and IoU cannot be computed there at all (finding F4).This notebook gets those numbers from a dataset that actually has voxel-level expertmasks, and produces a segmenter whose training we can describe honestly.**Run on:** Colab, free T4 GPU. Budget ~60-90 min end to end (download ~15 min,preprocessing ~10 min, training ~30 min).---### Why MSD Task06 and not LUNA16LUNA16's nodule annotations are **centroid + diameter** (`annotations.csv`; 1186nodules >=3 mm at 3-of-4 radiologist consensus). Scoring Dice there would meandrawing spherical pseudo-masks from those coordinates and then measuring oursegmenter against our own drawing - a circular number. LUNA16's real masks(`seg-lungs-LUNA16.zip`) segment **lung fields**, not lesions.MSD Task06_Lung has genuine per-voxel expert tumour masks: 96 thin-section CTvolumes of NSCLC patients from TCIA, of which the **64 training cases carry releasedlabels** (test labels are withheld for the challenge). Those 64 are what we use.LUNA16 is not dropped: it is used in the companion notebook for **detection**sensitivity/FROC, which is the metric its annotation format genuinely supports.### Why retrain rather than just evaluate the old checkpointWe have no training code and no training masks for `UNet_best_Model_checkpoint.h5`.Its provenance is unknown, which means it cannot be described honestly in a Methodsection. Retraining fixes that. The old checkpoint is still evaluated here forcomparison, as the first row of the segmentation ablation.### Two caveats that belong in the paper1. MSD Task06 labels are **tumour** masks (NSCLC). Our pipeline targets **nodules**.   Related, not identical.2. Dice is measured on MSD; classification is measured on IQ-OTH/NCCD. Different   domains (3D thin-section CT in Hounsfield units vs 8-bit JPEG-like slices). The   Dice number therefore does **not** transfer as a guarantee of segmentation quality   on IQ-OTH/NCCD, and the paper must say so. Section 9 does a qualitative transfer   check precisely so this is visible rather than assumed.

## 1. Environment

In [ ]:
import subprocess, sysfor pkg in ["nibabel", "opencv-python-headless", "tabulate"]:    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)import tensorflow as tf, kerasprint("tensorflow:", tf.__version__)print("keras     :", keras.__version__)gpus = tf.config.list_physical_devices("GPU")print("GPUs      :", gpus)assert gpus, "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

In [ ]:
import os, json, random, glob, shutil, timeimport numpy as npimport pandas as pdimport cv2import nibabel as nibimport matplotlib.pyplot as pltfrom tqdm.auto import tqdmimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersSEED = 42random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)# ---- configuration -------------------------------------------------------IMG_SIZE   = 256      # training resolution. 512 matches the deployed pipeline but                      # roughly triples training time; scoring is done at native                      # 512 either way by upsampling predictions.NEG_RATIO  = 1.0      # tumour-free slices kept per tumour-bearing sliceBATCH      = 8EPOCHS     = 40BASE_FILTERS = 32HU_LEVEL, HU_WIDTH = -600, 1500        # lung windowMIXED_PRECISION = True# --------------------------------------------------------------------------if MIXED_PRECISION:    keras.mixed_precision.set_global_policy("mixed_float16")    print("mixed precision:", keras.mixed_precision.global_policy().name)RESULTS_DIR = "/content/fyp_phase2s_results"os.makedirs(RESULTS_DIR, exist_ok=True)RESULTS = {"seed": SEED, "config": {    "img_size": IMG_SIZE, "neg_ratio": NEG_RATIO, "batch": BATCH, "epochs": EPOCHS,    "base_filters": BASE_FILTERS, "hu_level": HU_LEVEL, "hu_width": HU_WIDTH}}def save_json():    with open(f"{RESULTS_DIR}/results.json", "w") as f:        json.dump(RESULTS, f, indent=2, default=float)print("results ->", RESULTS_DIR)

## 2. Download MSD Task06_LungStreamed straight from the MONAI S3 mirror into `tar`, so the 9.2 GB archive is neverstored on disk. Only `imagesTr` and `labelsTr` are extracted (~5.7 GB); the unlabelled`imagesTs` is skipped. AppleDouble junk (`._*`) inside the archive is filtered out -those files are not valid NIfTI and will crash nibabel if left in.Takes roughly 10-20 min depending on Colab's link speed.

In [ ]:
MSD_URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"ROOT = "/content/msd"os.makedirs(ROOT, exist_ok=True)if not os.path.isdir(f"{ROOT}/Task06_Lung/labelsTr"):    cmd = (f"curl -L --retry 3 --fail '{MSD_URL}' | "           f"tar -x -C {ROOT} --wildcards --exclude='._*' "           f"'Task06_Lung/imagesTr/*' 'Task06_Lung/labelsTr/*'")    t0 = time.time()    subprocess.run(["bash", "-c", cmd], check=True)    print(f"downloaded + extracted in {(time.time()-t0)/60:.1f} min")else:    print("already present")IMG_DIR = f"{ROOT}/Task06_Lung/imagesTr"LBL_DIR = f"{ROOT}/Task06_Lung/labelsTr"print("images:", len(os.listdir(IMG_DIR)), "| labels:", len(os.listdir(LBL_DIR)))

### 2.1 Patient-level splitSplit **by volume (patient)**, never by slice. Adjacent axial slices of one patient arenear-duplicates; splitting at slice level leaks them across train and test and inflatesDice badly. It is a common bug in this literature, and having just documented anevaluation-leakage finding (F3) in Phase 0 we are not going to repeat it here.

In [ ]:
cases = sorted(f for f in os.listdir(LBL_DIR) if f.endswith(".nii.gz"))cases = [c for c in cases if os.path.exists(os.path.join(IMG_DIR, c))]print("labelled volumes:", len(cases))rng = np.random.RandomState(SEED)perm = rng.permutation(len(cases))n_te = max(1, int(round(0.20 * len(cases))))n_va = max(1, int(round(0.15 * len(cases))))test_c  = [cases[i] for i in perm[:n_te]]val_c   = [cases[i] for i in perm[n_te:n_te + n_va]]train_c = [cases[i] for i in perm[n_te + n_va:]]split_of = {**{c: "train" for c in train_c}, **{c: "val" for c in val_c},            **{c: "test" for c in test_c}}pd.DataFrame({"case": cases, "split": [split_of[c] for c in cases]}) \  .to_csv(f"{RESULTS_DIR}/msd_split_seed{SEED}.csv", index=False)print(f"train {len(train_c)} | val {len(val_c)} | test {len(test_c)}")print("test cases:", test_c)RESULTS["split"] = {"train": train_c, "val": val_c, "test": test_c}save_json()

## 3. PreprocessingMatched deliberately to the IQ-OTH/NCCD pipeline so the segmenter sees a comparablerepresentation in both domains:1. **HU windowing** to the lung window (level -600, width 1500) and rescale to 8-bit.2. **CLAHE** (clipLimit 2.0, 8x8 tiles) - the same step the IQ-OTH/NCCD pipeline uses.3. Resize to `IMG_SIZE`, then normalise to [-1, 1] as `(x - 127) / 127`.**Design choice worth stating in the paper:** a lung window saturates soft tissue, sotumour and mediastinum both render near-white and become harder to separate. Asoft-tissue window would segment better *on MSD* but would no longer match how the8-bit IQ-OTH/NCCD JPEGs are rendered, and transfer is the point here. Matching thetarget domain was chosen over maximising the MSD number.

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))def hu_to_uint8(sl):    lo, hi = HU_LEVEL - HU_WIDTH / 2.0, HU_LEVEL + HU_WIDTH / 2.0    return (np.clip((sl - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)def prep_slice(sl, size=None):    # HU slice -> uint8 CLAHE image at `size` (no normalisation yet)    size = size or IMG_SIZE    u8 = _clahe.apply(hu_to_uint8(sl))    if u8.shape != (size, size):        u8 = cv2.resize(u8, (size, size), interpolation=cv2.INTER_AREA)    return u8def to_model_input(u8):    return ((u8.astype(np.float32) - 127.0) / 127.0)[..., None]def load_volume(case):    img = nib.load(os.path.join(IMG_DIR, case))    lbl = nib.load(os.path.join(LBL_DIR, case))    v = np.asanyarray(img.dataobj, dtype=np.float32)    m = (np.asanyarray(lbl.dataobj) > 0).astype(np.uint8)    return v, m

In [ ]:
v, m = load_volume(train_c[0])print("case      :", train_c[0])print("volume    :", v.shape, v.dtype)print("HU range  :", float(v.min()), "..", float(v.max()))print("tumour vox:", int(m.sum()), f"({m.mean()*100:.4f}% of volume)")print("slices with tumour:", int((m.sum(axis=(0, 1)) > 0).sum()), "of", m.shape[2])

### 3.1 Diagnostic: does the legacy lung-ROI hack destroy tumour pixels?The original IQ-OTH/NCCD preprocessing applies a crude lung ROI before segmentation:inverse-threshold at 127, erode 4x4, dilate 13x13. Since we now have ground-truthtumour masks, we can measure for the first time what fraction of tumour pixels thatstep throws away. If the number is bad, the hack was actively hurting the pipeline anddropping it is a justified, measured change rather than a guess.

In [ ]:
def legacy_lung_roi(u8):    _, roi = cv2.threshold(u8, 127, 255, cv2.THRESH_BINARY_INV)    roi = cv2.erode(roi, np.ones([4, 4], np.uint8))    roi = cv2.dilate(roi, np.ones([13, 13], np.uint8))    return roi > 0kept, total, checked = 0, 0, 0for case in tqdm(train_c[:8], desc="ROI check"):    v, m = load_volume(case)    for z in np.where(m.sum(axis=(0, 1)) > 0)[0]:        u8 = prep_slice(v[:, :, z])        gt = cv2.resize(m[:, :, z], (IMG_SIZE, IMG_SIZE),                        interpolation=cv2.INTER_NEAREST) > 0        if gt.sum() == 0:            continue        kept += int((gt & legacy_lung_roi(u8)).sum()); total += int(gt.sum()); checked += 1frac = kept / max(total, 1)print(f"\ntumour-bearing slices checked : {checked}")print(f"tumour pixels surviving the ROI: {frac:.1%}")print("VERDICT:", "ROI is destructive - drop it" if frac < 0.80 else      "ROI is broadly safe, but it is still an untuned heuristic")RESULTS["legacy_roi_diagnostic"] = {"slices_checked": checked,                                    "tumour_pixel_retention": float(frac)}save_json()

## 4. Build the slice datasetEvery tumour-bearing slice is kept, plus `NEG_RATIO` tumour-free slices per positive,sampled at random. Tumours occupy a tiny fraction of each volume, so keeping allnegatives would give a ratio around 1:20 and the model would learn to predict emptymasks. The sampled ratio is recorded because it directly sets the class balance andtherefore has to appear in the paper.

In [ ]:
def build_slices(case_list, desc):    X, Y = [], []    for case in tqdm(case_list, desc=desc):        v, m = load_volume(case)        pos = np.where(m.sum(axis=(0, 1)) > 0)[0]        neg_pool = np.setdiff1d(np.arange(m.shape[2]), pos)        n_neg = min(len(neg_pool), int(round(NEG_RATIO * len(pos))))        neg = rng.choice(neg_pool, n_neg, replace=False) if n_neg else np.array([], int)        for z in np.concatenate([pos, neg]).astype(int):            X.append(prep_slice(v[:, :, z]))            Y.append(cv2.resize(m[:, :, z], (IMG_SIZE, IMG_SIZE),                                interpolation=cv2.INTER_NEAREST))        del v, m    return np.stack(X), np.stack(Y)Xtr, Ytr = build_slices(train_c, "train slices")Xva, Yva = build_slices(val_c, "val slices")print("train:", Xtr.shape, "positive slices:", int((Ytr.sum((1, 2)) > 0).sum()))print("val  :", Xva.shape, "positive slices:", int((Yva.sum((1, 2)) > 0).sum()))print("tumour pixel fraction (train):", f"{Ytr.mean():.5f}")RESULTS["dataset"] = {    "train_slices": int(len(Xtr)), "val_slices": int(len(Xva)),    "train_positive_slices": int((Ytr.sum((1, 2)) > 0).sum()),    "tumour_pixel_fraction_train": float(Ytr.mean())}save_json()

In [ ]:
def augment(x, y):    if tf.random.uniform([]) < 0.5:        x, y = tf.image.flip_left_right(x), tf.image.flip_left_right(y)    if tf.random.uniform([]) < 0.5:        x, y = tf.image.flip_up_down(x), tf.image.flip_up_down(y)    k = tf.random.uniform([], 0, 4, dtype=tf.int32)    return tf.image.rot90(x, k), tf.image.rot90(y, k)def make_ds(X, Y, training):    ds = tf.data.Dataset.from_tensor_slices((X, Y))    def cast(x, y):        return ((tf.cast(x, tf.float32) - 127.0) / 127.0)[..., None], \               tf.cast(y, tf.float32)[..., None]    ds = ds.map(cast, num_parallel_calls=tf.data.AUTOTUNE)    if training:        ds = ds.shuffle(2048, seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)ds_tr, ds_va = make_ds(Xtr, Ytr, True), make_ds(Xva, Yva, False)xb, yb = next(iter(ds_tr))print("batch:", xb.shape, xb.dtype, "| mask:", yb.shape, float(yb.numpy().mean()))

## 5. Model and lossA standard 4-level U-Net (Ronneberger et al., 2015), built here rather than inherited,so the architecture is fully documented in the paper.**Loss = BCE + soft Dice.** The original FYP used pure negative-Dice loss. With targetsthis sparse, pure Dice gives near-zero gradient whenever a batch happens to contain notumour pixels, which is a plausible contributor to the unstable training reported inthe FYP. The BCE term keeps gradients alive. This is a deliberate, reportable change.

In [ ]:
def conv_block(x, f):    for _ in range(2):        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)        x = layers.BatchNormalization()(x)        x = layers.Activation("relu")(x)    return xdef build_unet(size=IMG_SIZE, base=BASE_FILTERS):    inp = layers.Input((size, size, 1))    skips, x = [], inp    for i in range(4):        x = conv_block(x, base * 2 ** i)        skips.append(x)        x = layers.MaxPooling2D(2)(x)    x = conv_block(x, base * 16)    for i in reversed(range(4)):        x = layers.Conv2DTranspose(base * 2 ** i, 2, strides=2, padding="same")(x)        x = layers.Concatenate()([x, skips[i]])        x = conv_block(x, base * 2 ** i)    # float32 output is required under mixed precision for a numerically safe loss    out = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x)    return keras.Model(inp, out, name="unet")SMOOTH = 1.0def dice_coef(y_true, y_pred):    y_true = tf.cast(y_true, tf.float32); y_pred = tf.cast(y_pred, tf.float32)    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])    denom = tf.reduce_sum(y_true, [1, 2, 3]) + tf.reduce_sum(y_pred, [1, 2, 3])    return tf.reduce_mean((2. * inter + SMOOTH) / (denom + SMOOTH))def bce_dice_loss(y_true, y_pred):    bce = keras.losses.binary_crossentropy(y_true, y_pred)    return tf.reduce_mean(bce) + (1.0 - dice_coef(y_true, y_pred))model = build_unet()model.compile(optimizer=keras.optimizers.Adam(1e-3),              loss=bce_dice_loss, metrics=[dice_coef])print("params:", f"{model.count_params():,}")

In [ ]:
CKPT = f"{RESULTS_DIR}/unet_msd_best.weights.h5"cbs = [    keras.callbacks.ModelCheckpoint(CKPT, monitor="val_dice_coef", mode="max",                                    save_best_only=True, save_weights_only=True, verbose=1),    keras.callbacks.ReduceLROnPlateau(monitor="val_dice_coef", mode="max", factor=0.5,                                      patience=4, min_lr=1e-6, verbose=1),    keras.callbacks.EarlyStopping(monitor="val_dice_coef", mode="max", patience=10,                                  restore_best_weights=True, verbose=1),]t0 = time.time()hist = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS, callbacks=cbs, verbose=1)print(f"trained in {(time.time()-t0)/60:.1f} min")RESULTS["training"] = {"epochs_run": len(hist.history["loss"]),                       "best_val_dice_coef": float(max(hist.history["val_dice_coef"])),                       "minutes": (time.time() - t0) / 60}save_json()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val")ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()ax[1].plot(hist.history["dice_coef"], label="train")ax[1].plot(hist.history["val_dice_coef"], label="val")ax[1].set_title("soft Dice (batch-level)"); ax[1].set_xlabel("epoch"); ax[1].legend()plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_training_curves.png", dpi=150); plt.show()

## 6. Evaluation on held-out patientsTwo things done properly here, both of which change the number materially:- **Per-volume 3D Dice, not per-slice.** Averaging slice-level Dice inflates the score,  because tumour-free slices score ~1.0 for free under the smoothed formula. Dice is  computed once over each whole volume.- **Scored at native resolution.** Predictions are upsampled back to the volume's own  in-plane size before scoring, so `IMG_SIZE` is a training choice and not a way of  making the metric easier.The operating threshold is selected on **validation** and then applied unchanged totest. Tuning it on test would be a subtler version of the same leakage as F3.

In [ ]:
def predict_volume_probs(case, model_, size=None):    # Probability volume at the case's NATIVE in-plane resolution, plus its GT.    # Thresholding is left to the caller so a sweep costs one forward pass, not one    # per threshold.    size = size or IMG_SIZE    v, m = load_volume(case)    H, W, D = m.shape    probs = np.zeros((H, W, D), np.float32)    for s in range(0, D, 32):        zs = list(range(s, min(s + 32, D)))        batch = np.stack([to_model_input(prep_slice(v[:, :, z], size)) for z in zs])        p = model_.predict(batch, verbose=0)[..., 0].astype(np.float32)        for k, z in enumerate(zs):            probs[:, :, z] = cv2.resize(p[k], (W, H), interpolation=cv2.INTER_LINEAR)    del v    return probs, mdef predict_volume(case, thr, model_, size=None):    probs, m = predict_volume_probs(case, model_, size)    return (probs >= thr).astype(np.uint8), mdef volume_scores(pred, gt):    inter = float((pred & gt).sum()); ps, gs = float(pred.sum()), float(gt.sum())    union = ps + gs - inter    return {"dice": (2 * inter / (ps + gs)) if (ps + gs) > 0 else np.nan,            "iou": (inter / union) if union > 0 else np.nan,            "gt_voxels": gs, "pred_voxels": ps}

In [ ]:
THRS = [0.3, 0.4, 0.5, 0.6, 0.7]acc = {t: [] for t in THRS}for c in tqdm(val_c, desc="val volumes"):    probs, gt = predict_volume_probs(c, model)          # one forward pass per volume    for t in THRS:        acc[t].append(volume_scores((probs >= t).astype(np.uint8), gt)["dice"])    del probs, gtsweep = {t: float(np.nanmean(v)) for t, v in acc.items()}for t in THRS:    print(f"threshold {t:.1f} -> val mean volume Dice {sweep[t]:.4f}")BEST_THR = max(sweep, key=sweep.get)print("\nselected threshold (on VAL):", BEST_THR)RESULTS["threshold_sweep_val"] = sweepRESULTS["selected_threshold"] = BEST_THRsave_json()

In [ ]:
rows = []for c in tqdm(test_c, desc="test volumes"):    pred, gt = predict_volume(c, BEST_THR, model)    s = volume_scores(pred, gt); s["case"] = c    rows.append(s)test_df = pd.DataFrame(rows)[["case", "dice", "iou", "gt_voxels", "pred_voxels"]]print(test_df.round(4).to_string(index=False))summary = {"n_volumes": int(len(test_df)),           "dice_mean": float(test_df["dice"].mean()), "dice_std": float(test_df["dice"].std()),           "dice_median": float(test_df["dice"].median()),           "iou_mean": float(test_df["iou"].mean()), "iou_std": float(test_df["iou"].std()),           "iou_median": float(test_df["iou"].median()),           "threshold": BEST_THR}print("\n=== HELD-OUT TEST (per-volume, native resolution) ===")print(f"Dice  {summary['dice_mean']:.4f} +/- {summary['dice_std']:.4f}  (median {summary['dice_median']:.4f})")print(f"IoU   {summary['iou_mean']:.4f} +/- {summary['iou_std']:.4f}  (median {summary['iou_median']:.4f})")test_df.to_csv(f"{RESULTS_DIR}/test_per_volume.csv", index=False)RESULTS["test_summary"] = summaryRESULTS["test_per_volume"] = test_df.to_dict("records")save_json()

Report the **distribution**, not just the mean. Lung-tumour Dice is high-variance:a couple of tiny or atypical tumours can drag the mean down while the median staysrespectable, and a reviewer will want to see that rather than a single number.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))ax[0].boxplot([test_df["dice"].dropna(), test_df["iou"].dropna()], labels=["Dice", "IoU"])ax[0].set_ylim(0, 1); ax[0].set_title(f"per-volume scores (n={len(test_df)})")ax[0].grid(alpha=.3)ax[1].scatter(test_df["gt_voxels"], test_df["dice"], alpha=.75)ax[1].set_xscale("log"); ax[1].set_xlabel("tumour size (GT voxels, log)")ax[1].set_ylabel("Dice"); ax[1].set_title("Dice vs tumour size"); ax[1].grid(alpha=.3)plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_test_distribution.png", dpi=150); plt.show()

## 7. Baseline comparison: the original checkpointThe old `UNet_best_Model_checkpoint.h5` scored on the same held-out volumes, giving thefirst row of the segmentation ablation. Expect it to do poorly: it was trained onunknown data, for an unknown objective, in a different domain. That is the point ofthe comparison, and the number gets reported whatever it is.

In [ ]:
OLD_URL = ("https://github.com/haseebkhan9081/LViT_Vision_Transformer/"           "releases/download/weights-v1/UNet_best_Model_checkpoint.h5")OLD = "/content/UNet_best_Model_checkpoint.h5"try:    if not os.path.exists(OLD):        subprocess.run(["wget", "-q", "-O", OLD, OLD_URL], check=True)    from tensorflow.keras import backend as K    def _dc(y_true, y_pred):        f1, f2 = K.flatten(y_true), K.flatten(y_pred)        i = K.sum(f1 * f2)        return (2. * i + 1) / (K.sum(f1) + K.sum(f2) + 1)    old_model = tf.keras.models.load_model(        OLD, custom_objects={"dice_coef": _dc, "dice_coef_loss": lambda a, b: -_dc(a, b)},        compile=False)    old_size = old_model.input_shape[1] or 512      # None if the shape is undefined    print("old U-Net input:", old_model.input_shape, "-> using", old_size)    old_rows = [dict(volume_scores(*predict_volume(c, 0.5, old_model, old_size)), case=c)                for c in tqdm(test_c, desc="old checkpoint")]    old_df = pd.DataFrame(old_rows)    old_sum = {"dice_mean": float(old_df["dice"].mean()),               "dice_median": float(old_df["dice"].median()),               "iou_mean": float(old_df["iou"].mean()), "threshold": 0.5}    print(f"\nold checkpoint: Dice {old_sum['dice_mean']:.4f} "          f"(median {old_sum['dice_median']:.4f}) | IoU {old_sum['iou_mean']:.4f}")    RESULTS["old_checkpoint_on_msd"] = old_sumexcept Exception as e:    print("old-checkpoint comparison skipped:", type(e).__name__, e)    RESULTS["old_checkpoint_on_msd"] = {"error": f"{type(e).__name__}: {e}"}save_json()

## 8. Qualitative results

In [ ]:
show = test_c[:3]fig, axes = plt.subplots(len(show), 3, figsize=(10, 3.4 * len(show)))for row, case in zip(np.atleast_2d(axes), show):    pred, gt = predict_volume(case, BEST_THR, model)    z = int(np.argmax(gt.sum(axis=(0, 1))))          # slice with the most tumour    v, _ = load_volume(case)    base = prep_slice(v[:, :, z], gt.shape[0])    row[0].imshow(base, cmap="bone"); row[0].set_title(f"{case[:14]} z={z}", fontsize=9)    row[1].imshow(gt[:, :, z], cmap="gray"); row[1].set_title("ground truth", fontsize=9)    ov = cv2.cvtColor(base, cv2.COLOR_GRAY2RGB)    for cnts, col in [(cv2.findContours(gt[:, :, z], cv2.RETR_EXTERNAL,                                        cv2.CHAIN_APPROX_SIMPLE)[0], (0, 255, 0)),                      (cv2.findContours(pred[:, :, z], cv2.RETR_EXTERNAL,                                        cv2.CHAIN_APPROX_SIMPLE)[0], (255, 0, 0))]:        cv2.drawContours(ov, cnts, -1, col, 1)    row[2].imshow(ov); row[2].set_title("GT (green) vs pred (red)", fontsize=9)    for a in row: a.axis("off")plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_qualitative_msd.png", dpi=150); plt.show()

## 9. Transfer check on IQ-OTH/NCCD (qualitative only)The MSD-trained segmenter applied to the actual target domain. There is **no groundtruth here**, so no score is produced and none should be invented - this exists so thedomain gap is visible in the paper as a figure rather than being assumed away.Optional: needs `KAGGLE_API_TOKEN` (or the legacy `KAGGLE_USERNAME` / `KAGGLE_KEY`pair) in Colab Secrets. Skipped cleanly if absent.

In [ ]:
try:    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub"],                   check=False)    from google.colab import userdata    def _secret(name):        try:            v = userdata.get(name)            return v.strip() if v else None        except Exception:            return None    if _secret("KAGGLE_API_TOKEN"):                     # new-style KGAT_ token        os.environ["KAGGLE_API_TOKEN"] = _secret("KAGGLE_API_TOKEN")    else:                                               # legacy pair        os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME")        os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY")    import kagglehub    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")    cand = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]    IQ = os.path.dirname(cand[0])    picks = []    for folder in ["Bengin cases", "Malignant cases", "Normal cases"]:        fs = sorted(os.listdir(os.path.join(IQ, folder)))[:2]        picks += [(os.path.join(IQ, folder, f), folder) for f in fs]    fig, axes = plt.subplots(len(picks), 2, figsize=(7, 3.2 * len(picks)))    for row, (p, folder) in zip(np.atleast_2d(axes), picks):        g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)        # already 8-bit: skip HU windowing, keep CLAHE + resize so the rest matches        u8 = _clahe.apply(cv2.resize(g, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA))        pr = model.predict(to_model_input(u8)[None], verbose=0)[0, ..., 0]        mk = (pr >= BEST_THR).astype(np.uint8)        row[0].imshow(u8, cmap="bone"); row[0].set_title(folder, fontsize=9)        ov = cv2.cvtColor(u8, cv2.COLOR_GRAY2RGB)        cnts, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)        cv2.drawContours(ov, cnts, -1, (255, 0, 0), 1)        row[1].imshow(ov); row[1].set_title(f"pred ({mk.mean()*100:.2f}% area)", fontsize=9)        for a in row: a.axis("off")    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_transfer_iqothnccd.png", dpi=150)    plt.show()    print("NOTE: qualitative only. No ground truth exists here; do not report a score.")except Exception as e:    print("transfer check skipped:", type(e).__name__, e)

## 10. Bundle

In [ ]:
tbl = pd.DataFrame([    {"model": "Original UNet_best_Model_checkpoint.h5 (unknown training data)",     "dice_mean": RESULTS.get("old_checkpoint_on_msd", {}).get("dice_mean"),     "iou_mean": RESULTS.get("old_checkpoint_on_msd", {}).get("iou_mean")},    {"model": f"U-Net retrained on MSD Task06 ({IMG_SIZE}px, BCE+Dice)",     "dice_mean": RESULTS["test_summary"]["dice_mean"],     "iou_mean": RESULTS["test_summary"]["iou_mean"]},]).round(4)print(tbl.to_string(index=False))tbl.to_csv(f"{RESULTS_DIR}/segmentation_ablation.csv", index=False)with open(f"{RESULTS_DIR}/segmentation_ablation.md", "w") as f:    f.write(tbl.to_markdown(index=False))RESULTS["segmentation_ablation"] = tbl.to_dict("records")save_json()shutil.make_archive("/content/fyp_phase2s_results", "zip", RESULTS_DIR)print("\n", sorted(os.listdir(RESULTS_DIR)))try:    from google.colab import files    files.download("/content/fyp_phase2s_results.zip")except Exception as e:    print("download manually from the file browser:", e)

## 11. Send backAttach `fyp_phase2s_results.zip`. It contains `results.json`, the trained weights(`unet_msd_best.weights.h5`), the per-volume test CSV, the patient split, and all figures.Flag immediately if:- **Test Dice is near 0.** Check the training curves first. If validation Dice never  rose above ~0.1, the tumour signal is being lost in preprocessing - most likely the  lung window saturating soft tissue. Retry with a soft-tissue window  (`HU_LEVEL=40, HU_WIDTH=400`) and report both.- **The Colab session dies mid-training.** `unet_msd_best.weights.h5` is checkpointed  every time val Dice improves, so it survives. Copy it to Drive before rerunning.- **The download stalls.** The `curl | tar` stream has no resume. Rerun the cell; it  skips if `labelsTr` already exists.- **OOM during `build_slices`.** Lower `NEG_RATIO` to 0.5, or `IMG_SIZE` to 192.